## Week 1, Day 3 - Lab 2 (Free Models via Groq)

This is the community contribution version of Lab 2 using **only free models via Groq**.

We will:
1. Generate a challenging question using a Groq model
2. Have multiple free Groq models answer it
3. Use a judge model to score and rank the answers

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key found, begins: {groq_api_key[:7]}")
else:
    print("Groq API Key not set - add GROQ_API_KEY to your .env file")

In [ ]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)

## Step 1 - Generate a challenging question

We use `llama-3.3-70b-versatile` (Groq's most capable free model) to come up with a tough, thought-provoking question.

In [ ]:
question_prompt = """
Come up with a challenging, nuanced question that tests genuine reasoning ability.
It should NOT be a math puzzle. It should be a thought-provoking question requiring
intelligent insight. Include in the question that the answer must be kept to 1-2 sentences.
Only output the question itself, nothing else.
"""
messages = [
    {
        "role": "user",
        "content": question_prompt
    }
]
response = groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages= messages
)

question = response.choices[0].message.content
display(Markdown(f"**Question:** {question}"))

## Step 2 - Get answers from multiple free Groq models

We pit four different free models against each other:
- `llama-3.3-70b-versatile` — Meta's large capable model
- `llama3-8b-8192` — Meta's smaller fast model
- `mixtral-8x7b-32768` — Mistral's mixture-of-experts model
- `gemma2-9b-it` — Google's Gemma 2 model

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(f"### {model_name}\n{answer}"))

In [ ]:
model_name = "llama-3.3-70b-versatile"
response = groq.chat.completions.create(model=model_name, messages=messages)
record(model_name, response.choices[0].message.content)

In [ ]:
model_name = "groq/compound"
response = groq.chat.completions.create(model=model_name, messages=messages)
record(model_name, response.choices[0].message.content)

## Step 3 - Judge the answers

We ask `llama-3.3-70b-versatile` to act as an impartial judge and rank all answers.

In [ ]:
answers_text = ""
for i, (model, answer) in enumerate(zip(competitors, answers)):
    answers_text += f"\n\n**Model {i+1}: {model}**\n{answer}"

judge_prompt = f"""
You are an impartial judge evaluating AI responses to the following question:

{question}

Here are the answers from different models:
{answers_text}

Please rank the answers from best to worst. For each, give a score out of 10
and one sentence explaining why. End with a overall winner.
"""

judge_response = groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": judge_prompt}]
)

display(Markdown("## Judge Results\n" + judge_response.choices[0].message.content))